# AAROH - Phase 3: Simplification & Localization (RAG + LangGraph)

This notebook demonstrates the complete Phase 3 pipeline:
1. **LangGraph Agent Flow**: `cache_lookup → rag_retrieval → llm_simplification → cache_and_emit`
2. **RAG Context Retrieval**: Pulls the most relevant curriculum chunks from pgvector.
3. **Child-Friendly Simplification**: Converts textbook language into village analogies, micro-stories, and hands-on nature activities.
4. **Cultural Localization**: Tribal dialect adaptation (Gondi, Santhali, Chhattisgarhi, Bhili).
5. **Content Cache**: Two-tier cache (memory → JSON → DynamoDB) avoids redundant generation.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.aaroh.pipeline.simplification_localization_pipeline import SimplificationLocalizationPipeline
from src.aaroh.database.models import ConceptModel

pipeline = SimplificationLocalizationPipeline()
print('Pipeline initialized successfully!')

## 1. Direct Pipeline Run (with cache)

In [ ]:
result = pipeline.run_for_concept(
    concept_code='EVS-G3-WAT-01',
    target_language='hi',
    target_dialect='gon',
    grade_level=3,
    force_regenerate=True,
)

print(f'Concept: {result.concept_name}  |  Cached: {result.cached}')
print('\n--- Simplified Explanation ---')
print(result.simplified_explanation)
print('\n--- Story Narrative ---')
print(result.story_narrative)
print('\n--- Hands-On Activity ---')
print(result.hands_on_activity.get('title'))
print('Steps:', result.hands_on_activity.get('steps'))
print('\n--- Voice Script ---')
print(result.voice_script[:200], '...')

## 2. LangGraph Agent Flow (explicit graph traversal)

In [ ]:
lesson = pipeline.run_langgraph_for_concept(
    concept_code='EVS-G3-WAT-02',
    target_language='hi',
    target_dialect='sat',  # Santhali dialect
    grade_level=3,
    force_regenerate=True,
)
print('LangGraph lesson keys:', list(lesson.keys()))
print('\nReflection Questions:')
for q in lesson.get('reflection_questions', []):
    print(f'  Q: {q["question"]}')
    print(f'  Hint: {q["hint"]}')

## 3. Full Document Localization (All Concepts)

In [ ]:
from src.aaroh.database.models import DocumentModel
session = pipeline.db_manager.get_session()
docs = session.query(DocumentModel).all()
session.close()

if docs:
    doc_id = docs[0].id
    lessons = pipeline.run_for_document(doc_id, target_language='hi', target_dialect='hne')
    print(f'Localized {len(lessons)} concepts for document: {docs[0].title}')
    for l in lessons:
        print(f'  [{l.concept_code}] {l.concept_name} → activity: {l.hands_on_activity.get("title")}')

## 4. Content Cache Status

In [ ]:
cache = pipeline.cache
print(f'Total cached lessons: {len(cache.memory_cache)}')
for k, v in cache.memory_cache.items():
    print(f'  {k[:12]}... → concept={v["concept_code"]}, lang={v["target_language"]}, dialect={v["target_dialect"]}')